# 03 - Especialização com LoRA
**Requisito 05.** Ajuste fino do `Qwen2.5-1.5B-Instruct` com LoRA para: (1) responder **sempre em JSON válido**, (2) **citar o id do trecho** usado e (3) **recusar** quando o contexto não responde. O conhecimento continua vindo da base (RAG). Os dados de treino são gerados dos trechos e **não incluem** os casos de teste.
Use GPU (Colab T4). Dura ~20-40 min.

In [ ]:
# Setup (Colab: Ambiente de execução > Alterar tipo > GPU T4). Local: só rode a partir da raiz do repositório.
import os, sys, subprocess
REPO = "https://github.com/melo-zz/calcada-livre-genai.git"   # <- troque pelo link do repositório público da equipe
if os.path.exists("/content"):                                  # estamos no Colab
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO, "/content/repo"], check=True)
    os.chdir("/content/repo")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("pasta de trabalho:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass

In [ ]:
import os
from src.kb import carregar_trechos
from src.retrieval import Retriever
from src.lora_data import construir, salvar
os.makedirs(".cache", exist_ok=True)
trechos = carregar_trechos("base/trechos.jsonl")
r = Retriever(trechos, cache=".cache/emb.npy")
exemplos = construir(trechos, retriever=r, n_consta=360, n_nc=120, seed=SEED)
salvar(exemplos)
print(len(exemplos), "exemplos de treino (360 'consta' + 120 'não consta')")
print(exemplos[0]["messages"][-1]["content"][:400]); print("ALVO:", exemplos[0]["alvo"])

In [ ]:
from src.treino_lora import treinar
log = treinar(exemplos, saida="modelo/lora_qwen15b", epocas=2, seed=SEED)

In [ ]:
import matplotlib.pyplot as plt
loss = [(l["step"], l["loss"]) for l in log if "loss" in l]
plt.plot(*zip(*loss)); plt.xlabel("passo"); plt.ylabel("loss"); plt.title("Curva de treino do LoRA")
plt.savefig("avaliacao/loss_lora.png", dpi=150, bbox_inches="tight"); plt.show()
print("adaptador salvo em modelo/lora_qwen15b (não vai para o GitHub; reproduza com este notebook)")

Para usar o adaptador no notebook 04 em outra sessão do Colab, salve a pasta `modelo/lora_qwen15b` no Google Drive, ou rode o 03 e o 04 na mesma sessão.